# Fase 1 · Paso 4: piloto de localización tibiofemoral y recorte por rodilla

Esta libreta genera y muestra veinte recortes candidatos seudonimizados a partir de los diez estudios piloto. Exige que el paso 3 esté cerrado y sus parámetros congelados. No procesa la cohorte completa, no crea particiones, no entrena y no abre la prueba reservada.

## 1. Montar Drive y comprobar los cierres privados

In [ ]:
import json
import os
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
private_root = userdata.get('KNEE_DATA_ROOT') or os.environ.get('KNEE_DATA_ROOT')
if not private_root:
    raise RuntimeError('Configura el secreto KNEE_DATA_ROOT antes de continuar.')
os.environ['KNEE_DATA_ROOT'] = private_root
private_root_path = Path(private_root)
bilateral_dir = private_root_path / 'outputs' / 'auditorias' / 'fase_1' / 'paso_3_separacion_bilateral'
bilateral_closure = bilateral_dir / 'cierre_paso_3_publico.json'
frozen_parameters = bilateral_dir / 'parametros_congelados.json'
if not bilateral_closure.is_file() or not frozen_parameters.is_file():
    raise FileNotFoundError('No se encontró el cierre congelado del paso 3.')
closure = json.loads(bilateral_closure.read_text(encoding='utf-8'))
if closure.get('status') != 'closed' or not closure.get('parameters_frozen'):
    raise RuntimeError('El paso 3 todavía no está cerrado técnicamente.')
print('Drive montado y cierre congelado del paso 3 localizado.')

## 2. Obtener una revisión identificable del repositorio

In [ ]:
import subprocess

REPO_URL = 'https://github.com/AlonsoYess/Knee.git'
BRANCH = 'codex/fase-0-base-reproducible'
REPO_DIR = Path('/content/Knee_fase1_paso4')

if REPO_DIR.exists():
    if not (REPO_DIR / '.git').is_dir():
        raise RuntimeError('La ruta de trabajo existe y no es un clon Git; usa un entorno nuevo.')
    dirty = subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout
    if dirty:
        raise RuntimeError('El clon temporal contiene cambios. Restablece el entorno de ejecución.')
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'merge', '--ff-only', f'origin/{BRANCH}'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
print('Revisión preparada:', commit)

## 3. Instalar y comprobar el código

In [ ]:
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_DIR, check=True)
if subprocess.run(['git', 'status', '--porcelain'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout:
    raise RuntimeError('Las comprobaciones modificaron el repositorio temporal.')
print('Código y pruebas automáticas preparados.')

## 4. Generar los veinte recortes candidatos del piloto

In [ ]:
config_path = REPO_DIR / 'configs' / 'joint_localization.example.json'
subprocess.run([sys.executable, '-m', 'knee.joint_localization', '--config', str(config_path)], cwd=REPO_DIR, check=True)

## 5. Mostrar las vistas ciegas para revisión

In [ ]:
from IPython.display import Image as IPImage, display

output_dir = private_root_path / 'outputs' / 'auditorias' / 'fase_1' / 'paso_4_localizacion_tibiofemoral'
preview_dir = output_dir / 'previews_ciegas'
previews = sorted(preview_dir.glob('case_*_localizacion.png'))
if len(previews) != 20:
    raise RuntimeError(f'Se esperaban 20 vistas y se encontraron {len(previews)}.')
for preview in previews:
    print(preview.stem)
    display(IPImage(filename=str(preview), width=800))
print('Revisión pendiente de documentar en:', output_dir / 'revision_visual_ciega.csv')

## 6. Registrar la preparación controlada

In [ ]:
from datetime import datetime, timezone

summary_path = output_dir / 'resumen_localizacion_publico.json'
summary = json.loads(summary_path.read_text(encoding='utf-8'))
if summary.get('status') != 'ready_for_blinded_review':
    raise RuntimeError('El piloto de localización no quedó listo para revisión.')
record = {
    'phase': 1,
    'step': 4,
    'operation': 'prepare_blinded_tibiofemoral_localization_pilot',
    'executed_at_utc': datetime.now(timezone.utc).isoformat(),
    'git_commit': commit,
    'status': summary['status'],
    'algorithm_version': summary['algorithm_version'],
    'processed_unique_studies': summary['processed_unique_studies'],
    'processed_knees': summary['processed_knees'],
    'visual_review_status': 'pending',
    'mass_processing_executed': False,
    'partitions_created': False,
    'training_executed': False,
    'reserved_test_opened': False,
}
(output_dir / 'registro_preparacion.json').write_text(json.dumps(record, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')
print('Fase 1, paso 4 preparado para revisión visual ciega. El paso aún no está cerrado.', record)